# Capstone — mirrors your deployed research paper

This notebook brings together the complete work from the previous weeks into one final research artifact.

It summarizes the research question, data, methodology, model results, limitations, ranked recommendations, and the artifacts that support the final paper.

The goal is to present the work clearly, honestly, and reproducibly using observed and measured results.

## 1. Question
## Research Question

This project investigates which content pages should be prioritized for review based on their observed search-performance signals.

The main question is:

**Can a simple machine-learning model identify high-performing and lower-performing content more effectively than the Week-4 baseline rule?**

The decision supported by this analysis is content prioritization. The output is intended to help a content team decide which pages may need review, monitoring, or further investigation.

This analysis provides decision support only. It does not prove that changing a page will improve its future search performance.

In [1]:
import pandas as pd
import numpy as np

# Load the repository CSV dataset
data_path = "../../data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nResearch question:")
print("Can a machine-learning model identify content performance patterns")
print("more effectively than the Week-4 baseline rule?")

Dataset loaded successfully.
Rows: 30000
Columns: 44

Research question:
Can a machine-learning model identify content performance patterns
more effectively than the Week-4 baseline rule?


## 2. Data

The analysis uses the anonymized CSV provided inside the internship repository:

`data/raw/content_refresh_anonymized.csv`

I did not use the Hugging Face warehouse dataset because I was unable to reliably access it. Instead, I used the repository CSV consistently for the remaining analysis.

The dataset contains content-level search and engagement signals such as search volume, impressions, clicks, sessions, CTR, average position, engagement rate, content age, freshness information, and AI traffic.

Private client names, URLs, and private search queries were not used.

Fields that represent future outcomes or could directly reveal the prediction target were excluded from the model features. The analysis therefore focuses on signals that are available in the supplied dataset and can be used for decision-support analysis.

In [2]:
# Basic dataset information

print("Total rows:", len(df))
print("Total columns:", len(df.columns))

print("\nDataset columns:")
print(df.columns.tolist())

# Show missing values
missing_values = df.isnull().sum()

print("\nColumns with missing values:")
display(
    missing_values[missing_values > 0]
    .sort_values(ascending=False)
    .head(15)
)

# Show the main fields used in the analysis
candidate_columns = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "content_age_days",
    "days_since_last_update"
]

available_columns = [
    col for col in candidate_columns
    if col in df.columns
]

print("\nAvailable analysis fields:")
print(available_columns)

display(df[available_columns].head())

Total rows: 30000
Total columns: 44

Dataset columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Columns with missing values:


provider_used        21438
word_count_tier       7699
char_count            7699
word_count            7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125
dtype: int64


Available analysis fields:
['search_volume', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'engaged_sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'content_age_days', 'days_since_last_update']


,search_volume,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,engaged_sessions_90d,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,content_age_days,days_since_last_update
0,10.0,3803,29,22,17,1,0.76,10.6,5.88,4.55,0.0,187,20
1,90.0,15320,7,10,9,0,0.05,20.3,0.00,10.00,0.0,445,25
2,0.0,12581,11,14,11,0,0.09,36.5,0.00,28.57,0.0,141,20
3,10.0,11751,58,87,78,1,0.49,6.2,1.28,3.45,0.0,463,22
4,0.0,19140,24,177,145,0,0.13,44.0,0.00,24.29,0.0,263,14


## 3. Methodology

The analysis uses a supervised classification approach.

The target is based on observed `clicks_90d`. Pages with clicks above the dataset median are treated as the higher-performance class, while pages at or below the median are treated as the lower-performance class.

The model uses numeric search and content signals that are available in the repository CSV. Missing numeric values are filled using the training-data median.

A Logistic Regression model is used because it is relatively simple, interpretable, and provides a useful comparison with the transparent Week-4 baseline.

The evaluation uses a fixed train/test split so that the model and baseline can be compared on the same held-out observations.

The feature set does not intentionally include the target column or future outcome windows.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Create the observed-performance label
median_clicks = df["clicks_90d"].median()

df["performance_label"] = (
    df["clicks_90d"] > median_clicks
).astype(int)

print("Median clicks:", median_clicks)
print("\nLabel distribution:")
print(df["performance_label"].value_counts())

# Features that exist in the repository CSV
feature_columns = [
    "search_volume",
    "impressions_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

feature_columns = [
    col for col in feature_columns
    if col in df.columns
]

X = df[feature_columns].copy()
y = df["performance_label"].copy()

# Keep only numeric features
X = X.select_dtypes(include="number")

# Fill missing values
X = X.fillna(X.median())

# Same fixed split for model evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Logistic Regression pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

print("\nFeatures used:")
print(X.columns.tolist())

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

Median clicks: 1.0

Label distribution:
performance_label
0    17025
1    12975
Name: count, dtype: int64

Features used:
['search_volume', 'impressions_90d', 'pageviews_90d', 'sessions_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Training rows: 24000
Testing rows: 6000


## 4. Results (vs baseline)

The model is compared with the Week-4 baseline using the same held-out test set.

The baseline is a transparent rule-based score intended to prioritize content for review using observed search-performance signals. The model provides a different way of ranking the same type of content.

The comparison is descriptive and directional. A higher test score indicates better measured performance on this particular split; it does not establish that the model will always outperform the baseline on new data.

In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report

# Model predictions
model_pred = model.predict(X_test)
model_prob = model.predict_proba(X_test)[:, 1]

# Create a simple transparent baseline

baseline_df = df.loc[X_test.index].copy()

baseline_score = pd.Series(
    0.0,
    index=baseline_df.index
)

# Higher search volume = more opportunity
if "search_volume" in baseline_df.columns:
    volume_median = df["search_volume"].median()
    baseline_score += (
        baseline_df["search_volume"] > volume_median
    ).astype(float)

# Lower CTR = possible improvement opportunity
if "ctr" in baseline_df.columns:
    ctr_median = df["ctr"].median()
    baseline_score += (
        baseline_df["ctr"] < ctr_median
    ).astype(float)

# Older content = possible refresh opportunity
if "days_since_last_update" in baseline_df.columns:
    freshness_median = df["days_since_last_update"].median()
    baseline_score += (
        baseline_df["days_since_last_update"] > freshness_median
    ).astype(float)

# Convert baseline score into a binary prediction
baseline_pred = (
    baseline_score >= baseline_score.median()
).astype(int)

# Compare model and baseline

results = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, model_pred)
    ],
    "Precision": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, model_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred, zero_division=0),
        recall_score(y_test, model_pred, zero_division=0)
    ],
    "F1": [
        f1_score(y_test, baseline_pred, zero_division=0),
        f1_score(y_test, model_pred, zero_division=0)
    ]
})

display(results.round(4))

print("\nModel classification report:")
print(
    classification_report(
        y_test,
        model_pred,
        zero_division=0
    )
)

,Method,Accuracy,Precision,Recall,F1
0,Week-4 baseline,0.3397,0.3614,0.6867,0.4736
1,Logistic Regression,0.8812,0.8670,0.8566,0.8618



Model classification report:
              precision    recall  f1-score   support

           0       0.89      0.90      0.90      3405
           1       0.87      0.86      0.86      2595

    accuracy                           0.88      6000
   macro avg       0.88      0.88      0.88      6000
weighted avg       0.88      0.88      0.88      6000



## 5. Limitations

This analysis has several important limitations.

First, the analysis uses the repository CSV rather than the full Hugging Face warehouse, so the findings should not be treated as representative of the entire production dataset.

Second, the performance label is derived from observed clicks in the supplied data. Therefore, the model identifies patterns associated with observed performance rather than proving future performance.

Third, search signals such as CTR, position, impressions, and clicks are related to one another, so the model may capture existing relationships rather than independent causal effects.

The baseline and model are intended for decision-support and prioritization. They should not automatically change, publish, remove, or rewrite content.

Human review is required before taking action on individual pages.

In [5]:
# Simple limitation and safety checks

print("Dataset used:")
print("../../data/raw/content_refresh_anonymized.csv")

print("\nTarget column:")
print("performance_label")

print("\nExcluded directly from model features:")
excluded_columns = [
    "clicks_90d",
    "performance_label"
]

print(excluded_columns)

print("\nModel feature count:", len(X.columns))

print("\nImportant limitation:")
print(
    "The model measures patterns in the supplied dataset and does not "
    "prove future causal improvement."
)

print("\nHuman review is required before content actions are taken.")

Dataset used:
../../data/raw/content_refresh_anonymized.csv

Target column:
performance_label

Excluded directly from model features:
['clicks_90d', 'performance_label']

Model feature count: 16

Important limitation:
The model measures patterns in the supplied dataset and does not prove future causal improvement.

Human review is required before content actions are taken.


## 6. Ranked recommendations

The ranked recommendations are intended to turn the analysis into a practical content-review queue.

Higher-ranked pages should be reviewed first when their observed signals indicate a stronger opportunity for investigation.

The recommendation is decision-support only. A high score does not mean that a page is definitely performing poorly or that a refresh will definitely improve performance.

Human reviewers should consider the page's actual content, search intent, business context, and recent changes before taking action.

In [6]:
# Create a ranked recommendation queue from the available signals

recommendations = df.copy()

recommendations["action_score"] = 0.0

# High search volume increases review priority
if "search_volume" in recommendations.columns:
    volume_median = recommendations["search_volume"].median()
    recommendations["action_score"] += (
        recommendations["search_volume"] > volume_median
    ).astype(float)

# Low CTR can indicate an opportunity for review
if "ctr" in recommendations.columns:
    ctr_median = recommendations["ctr"].median()
    recommendations["action_score"] += (
        recommendations["ctr"] < ctr_median
    ).astype(float)

# Older pages can be candidates for refresh review
if "days_since_last_update" in recommendations.columns:
    freshness_median = recommendations["days_since_last_update"].median()
    recommendations["action_score"] += (
        recommendations["days_since_last_update"] > freshness_median
    ).astype(float)

# Reason code and action
def assign_reason(row):
    if (
        "search_volume" in recommendations.columns
        and row["search_volume"] > volume_median
        and "ctr" in recommendations.columns
        and row["ctr"] < ctr_median
    ):
        return "HIGH_VOLUME_LOW_CTR"
    
    if (
        "days_since_last_update" in recommendations.columns
        and row["days_since_last_update"] > freshness_median
    ):
        return "STALE_CONTENT"
    
    return "MONITOR"

def assign_action(row):
    if row["reason_code"] == "HIGH_VOLUME_LOW_CTR":
        return "Review CTR opportunity"
    
    if row["reason_code"] == "STALE_CONTENT":
        return "Review for refresh"
    
    return "Monitor"

recommendations["reason_code"] = recommendations.apply(
    assign_reason,
    axis=1
)

recommendations["action"] = recommendations.apply(
    assign_action,
    axis=1
)

# Rank the queue
recommendations = recommendations.sort_values(
    by="action_score",
    ascending=False
).reset_index(drop=True)

recommendations["rank"] = recommendations.index + 1

# Display the top recommendations
display(
    recommendations[
        [
            "rank",
            "content_id",
            "action_score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

,rank,content_id,action_score,reason_code,action
0,1,content_3b6a4a9b18b8,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
1,2,content_e68dcaf84daf,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
2,3,content_3596dbcf83da,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
3,4,content_3df29e6ccc87,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
4,5,content_fb3e6fee9571,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
5,6,content_33f886c6b765,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
6,7,content_0c957eac7ed9,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
7,8,content_c8630f323532,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
8,9,content_da476514f660,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
9,10,content_26e7e98c8e41,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity


## 7. Artifacts the paper embeds

The final paper should use the reproducible outputs generated by the project.

The main artifacts are the executed notebooks covering the data contract, feature and leakage checks, signal audit, baseline, model, validation audit, and action playbook.

The final research paper should reference the measured model-versus-baseline results and the ranked recommendation output rather than manually entering unsupported numbers.

The analysis remains reproducible because the notebook reads the repository CSV and performs the calculations from the data.

In [7]:
from pathlib import Path

# List the main project notebooks
notebook_paths = [
    "../../work/notebooks/w03_feature_leakage_check.ipynb",
    "../../work/notebooks/w04_signal_audit.ipynb",
    "../../work/notebooks/w04_baseline_score.ipynb",
    "../../work/notebooks/w05_model.ipynb",
    "../../work/notebooks/w06_validation_audit.ipynb",
    "../../work/notebooks/w07_action_playbook.ipynb",
    "../../work/notebooks/capstone.ipynb"
]

print("Project artifacts:")

for path in notebook_paths:
    exists = Path(path).exists()
    print(f"{path} -> {'FOUND' if exists else 'NOT FOUND'}")

# Show the main result table again for the paper
print("\nModel vs baseline results:")
display(results.round(4))

print("\nTop ranked recommendations:")
display(
    recommendations[
        [
            "rank",
            "content_id",
            "action_score",
            "reason_code",
            "action"
        ]
    ].head(10)
)

Project artifacts:
../../work/notebooks/w03_feature_leakage_check.ipynb -> FOUND
../../work/notebooks/w04_signal_audit.ipynb -> FOUND
../../work/notebooks/w04_baseline_score.ipynb -> FOUND
../../work/notebooks/w05_model.ipynb -> FOUND
../../work/notebooks/w06_validation_audit.ipynb -> FOUND
../../work/notebooks/w07_action_playbook.ipynb -> FOUND
../../work/notebooks/capstone.ipynb -> FOUND

Model vs baseline results:


,Method,Accuracy,Precision,Recall,F1
0,Week-4 baseline,0.3397,0.3614,0.6867,0.4736
1,Logistic Regression,0.8812,0.8670,0.8566,0.8618



Top ranked recommendations:


,rank,content_id,action_score,reason_code,action
0,1,content_3b6a4a9b18b8,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
1,2,content_e68dcaf84daf,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
2,3,content_3596dbcf83da,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
3,4,content_3df29e6ccc87,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
4,5,content_fb3e6fee9571,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
5,6,content_33f886c6b765,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
6,7,content_0c957eac7ed9,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
7,8,content_c8630f323532,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
8,9,content_da476514f660,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity
9,10,content_26e7e98c8e41,3.0,HIGH_VOLUME_LOW_CTR,Review CTR opportunity


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.